In [1]:
import torch 

/home/mnl/Desktop/University/Fall 2025-2026/fyp/K-KANs/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


### True Data Generation Process 

#### lloop

In [ ]:
torch.manual_seed(0)
n = 150
p = 10 #NOTE: so that the true covariance matrix is not full rank (since n > p) (if we use a linear kernel without a jitter)

## we do not really care where X comes from
X_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc= torch.zeros(p) , # mean vector of zeros
    covariance_matrix=torch.eye(p)  # identity covariance matrix
).sample((n,))  # n x p

## what we care about is that the vector (y_1, ..., y_n)^T comes from a guassian process with a certain kernel (that depends on X)
## in this example I will be taking that k(x_i,x_j) = linear kernel = x_i^T * x_j, where the sigma^2 = 1
true_mean = torch.zeros(n)  # n x 1
true_cov = torch.eye(n)  # n x n
true_sigma_squared = 1.0 
l_for_rbf = 1.0  # length scale for rbf kernel
for i in range(n):
    for j in range(n):
        # rbf kernel
        # true_cov[i, j] = true_sigma_squared * torch.exp(-torch.norm(X_data[i] - X_data[j])**2 / (2 * (l_for_rbf ** 2))) #NOTE: even if I used this kernel, the determinant is being 0, but the rank is full.
        true_cov[i, j] = true_sigma_squared * (X_data[i].unsqueeze(0) @ X_data[j].unsqueeze(1))   # linear kernel, we willdo  unsqueeze to add a dimension for matrix multiplication, resulting in a (1 x p) @ (p x 1) = (1 x 1) tensor
        # adding a small value to the diagonal to ensure positive definiteness #NOTE(see README for more details)
        # true_cov[i, i] += torch.distributions.normal.Normal(0, 0.001).sample()
        true_cov[i, i] += 1e-5 #NOTE this jitter is very weird, I am not sure why it is working, #NOTE: not always it works

rank = torch.linalg.matrix_rank(true_cov)
print(f"rank(true_cov): {rank}, expected: {n}")  # should be n
print(f"Determinant(true_cov): {torch.linalg.det(true_cov)}, expected: non-zero" )  # should be non-zero, #NOTE: if we get 0 this MIGHT mean underflow 

## by the assumption of the guassian process, the vector (f(x_1), ..., f(x_n))^T follows a multivariate normal distribution with mean 0 and covariance matrix K (n x n)
## so we can sample from that distribution to get f_data = (f(x_1), ..., f(x_n))^T
f_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc=true_mean.squeeze(),  # n (not n x 1)
    covariance_matrix=true_cov
).sample()  # n x 1
# make f_data of shape (n, 1)
f_data = f_data.unsqueeze(1)  # n x 1
print(f"Shape of f_data: {f_data.shape}, expected: ({n}, 1)")  # should be (n, 1)

## for this first experiment, we will assume y = f(x) without noise
y_data = f_data  # n x 1
print(f"Shape of y_data: {y_data.shape}, expected: ({n}, 1)")  # should be (n, 1)

#### matrix

In [41]:
torch.manual_seed(0)
n = 150
p = 10 #NOTE: so that the true covariance matrix is not full rank (since n > p) (if we use a linear kernel without a jitter)

## we do not really care where X comes from
X_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc= torch.zeros(p) , # mean vector of zeros
    covariance_matrix=torch.eye(p)  # identity covariance matrix
).sample((n,))  # n x p

## what we care about is that the vector (y_1, ..., y_n)^T comes from a guassian process with a certain kernel (that depends on X)
## in this example I will be taking that k(x_i,x_j) = linear kernel = x_i^T * x_j, where the sigma^2 = 1
true_mean = torch.zeros(n)  # n x 1
true_cov = torch.eye(n)  # n x n

true_cov = X_data @ X_data.T
true_cov = true_cov + 1e-2 * torch.eye(n) #NOTE: jitter depends on the daa, plus here we care about checking the rank and not the determinant(becuase it is unstable so we might go underflow and have a full rank matrix with "0" determinant)

rank = torch.linalg.matrix_rank(true_cov)
print(f"rank(true_cov): {rank}, expected: {n}")  # should be n
print(f"Determinant(true_cov): {torch.linalg.det(true_cov)}, expected: non-zero" )  # should be non-zero, #NOTE: if we get 0 this MIGHT mean underflow 

## by the assumption of the guassian process, the vector (f(x_1), ..., f(x_n))^T follows a multivariate normal distribution with mean 0 and covariance matrix K (n x n)
## so we can sample from that distribution to get f_data = (f(x_1), ..., f(x_n))^T
f_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc=true_mean.squeeze(),  # n (not n x 1)
    covariance_matrix=true_cov
).sample()  # n x 1
# make f_data of shape (n, 1)
f_data = f_data.unsqueeze(1)  # n x 1
print(f"Shape of f_data: {f_data.shape}, expected: ({n}, 1)")  # should be (n, 1)

## for this first experiment, we will assume y = f(x) without noise
y_data = f_data  # n x 1
print(f"Shape of y_data: {y_data.shape}, expected: ({n}, 1)")  # should be (n, 1)
print(f"True Covariance Matrix is : {true_cov}")

rank(true_cov): 150, expected: 150
Determinant(true_cov): 0.0, expected: non-zero
Shape of f_data: torch.Size([150, 1]), expected: (150, 1)
Shape of y_data: torch.Size([150, 1]), expected: (150, 1)
True Covariance Matrix is : tensor([[10.3296,  2.6554, -1.2587,  ..., -3.7375,  1.9617, -3.1413],
        [ 2.6554,  8.7385,  4.9349,  ...,  0.0365,  1.3584, -1.9529],
        [-1.2587,  4.9349, 11.7532,  ...,  1.1796,  3.6135, -3.4295],
        ...,
        [-3.7375,  0.0365,  1.1796,  ..., 18.2683,  1.4861,  4.7580],
        [ 1.9617,  1.3584,  3.6135,  ...,  1.4861,  9.8168, -1.7035],
        [-3.1413, -1.9529, -3.4295,  ...,  4.7580, -1.7035,  7.1653]])


### Train Loop: Maybe we need to  Optimize log-sigmas instead of sigmas directly

In [45]:
# now my goal is to maximize the marginal log likelihood, so I will define my loss = - MLL (because we care to maximize MLL which is equivalent to minimizing - MLL)

## intialization of the kernel matrix
# sigmas_pred = torch.randn(p,1, requires_grad=True) + torch.tensor(1.0)
sigmas_pred = torch.distributions.normal.Normal(1.0, 0.1).sample((p,1))
sigmas_pred.requires_grad_(True)
noise_logvar = torch.tensor(-4.0, requires_grad=True)
learning_rate = 0.001
epochs = 500
losses = []

for epoch in range(epochs): 
    # linear_kernel_pred = torch.ones(n,n) # these we do not care about their value they are just placeholder for now #TODO: investigate if the gradients will track chnages from one to k_i_j or not ( I think no )
    # instead of this loop;;;; I will do matrix multiplication 
    diag = torch.diag(sigmas_pred.squeeze()**2)      # p × p
    # K = X_data @ diag @ X_data.T   
    # K = K + 1e-3 * torch.eye(n) # n x n, #NOTE: why the jitter works ?
    

    K = X_data @ diag @ X_data.T
    K = K + torch.exp(noise_logvar) * torch.eye(n)

    # print(f"Rank of predicted kernel at epoch {epoch} is {torch.linalg.matrix_rank(linear_kernel_pred)}, expected: {n}")  # should be n
    # print(f"Determinant of predicted kernel at epoch {epoch} is {torch.linalg.det(linear_kernel_pred)}, expected: non-zero")  # should be non-zero
    # Computation of the loss 
    # K_inv = torch.linalg.inv(K)
    mu = torch.zeros(n,1) #TODO: how can we make it depend also on X_data
    sub = y_data - mu
    # #NOTE: if we do it this way it will become infinity becasue of the exponentiation
    # # num = torch.exp(-0.5* (sub.T @ K_inv @ sub)) 
    # # den = torch.sqrt((2*torch.pi)**n * torch.det(K))  
    # # MLL = num/den
    # # loss = - torch.log(MLL + 1e-10)  # adding a small value to avoid log(0) 
    # quad = (sub.T @ K_inv @ sub).squeeze()            # (y-mu)^T K^{-1} (y-mu)
    sign, logdet = torch.linalg.slogdet(K)
    if sign <= 0:
        print("K is not PD at epoch", epoch)
        break
    
    # stable quadratic form
    quad = (sub.T @ torch.linalg.solve(K, sub)).squeeze()
    log_mll = -0.5 * quad - 0.5 * logdet - 0.5 * n * torch.log(torch.tensor(2 * torch.pi))
    loss = -log_mll
    losses.append(loss.item())

    loss.backward()
    #print(f" Loss at epoch {epoch} is {loss.item()}")
    # update sigmas_pred
    grad_norm = sigmas_pred.grad.norm().item()
    noise_grad_norm = noise_logvar.grad.norm().item()
    with torch.no_grad():
        sigmas_pred -= learning_rate * sigmas_pred.grad # p x 1 
        sigmas_pred.grad.zero_()

        noise_logvar -= learning_rate * noise_logvar.grad # scalar
        noise_logvar.grad.zero_()

    if epoch % 10 == 0: 
        # print(f"At epoch {epoch}, Covariance Matrix Predicted is : {K}")
        print(f"Loss at epoch {epoch} is {loss.item()}")
        print(f" Probaility at epoch {epoch} is {torch.exp(-loss).item()}")
        print(f"  grad norm = {grad_norm:.3e}")
        print(f"  noise grad norm = {noise_grad_norm:.3e}")
        print(f"  sigmas_pred = {sigmas_pred.view(-1)[:5].data}")  # first 5
print(f" Losses over epochs: {losses}")


Loss at epoch 0 is -78.69879150390625
 Probaility at epoch 0 is 1.5081720749590527e+34
  grad norm = 2.146e+00
  noise grad norm = 3.637e+01
  sigmas_pred = tensor([1.2017, 0.9215, 1.0262, 1.0772, 1.0676])
Loss at epoch 10 is -88.17478942871094
 Probaility at epoch 10 is 1.9670911180029255e+38
  grad norm = 2.099e+00
  noise grad norm = 2.423e+01
  sigmas_pred = tensor([1.1966, 0.9277, 1.0385, 1.0794, 1.0622])
Loss at epoch 20 is -92.01954650878906
 Probaility at epoch 20 is inf
  grad norm = 2.062e+00
  noise grad norm = 1.436e+01
  sigmas_pred = tensor([1.1915, 0.9336, 1.0502, 1.0816, 1.0569])
Loss at epoch 30 is -93.30281066894531
 Probaility at epoch 30 is inf
  grad norm = 2.021e+00
  noise grad norm = 7.801e+00
  sigmas_pred = tensor([1.1865, 0.9393, 1.0613, 1.0837, 1.0516])
Loss at epoch 40 is -93.6927490234375
 Probaility at epoch 40 is inf
  grad norm = 1.990e+00
  noise grad norm = 4.013e+00
  sigmas_pred = tensor([1.1814, 0.9448, 1.0719, 1.0857, 1.0463])
Loss at epoch 50 is 